<a href="https://colab.research.google.com/github/Zeeshan4511/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeeshan4511/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Very high model performance

The FlyRank research paper reported very high model performance. The Logistic Regression model achieved an F1-score of 99.87%, accuracy of 99.85%, and ROC-AUC of 1.000. The other tested models also produced very high scores.

**My methodology question:**
I would respectfully ask whether the validation design tests the model on truly independent examples. The paper used a stratified random train-test split. I would want to check whether related observations could appear in both the training and testing data. If related records are present in both sets, the measured performance could be more optimistic than performance on completely unseen groups.

### Finding 2 — The target label was synthetically created

The paper explains that the original dataset did not contain the required lung cancer risk target. Instead, a RiskScore was created using features such as Age, Tumour Size, Smoking Pack-Years, Family History, and Chronic Lung Disease. A threshold was then used to create the final risk labels.

**My methodology question:**
I would ask how independent the target label is from the features used by the model. Some of the same features used to create the target label are also available as model inputs. This means the model can learn patterns that are directly related to the formula used to generate the label. Therefore, the very high measured performance should be understood as performance on a synthetic target rather than evidence of real-world clinical prediction.

The paper also acknowledges that the synthetic labels may create an artificially easy classification problem. This makes careful validation and safe claim language especially important.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

df = pd.read_csv("/crc_dataset.csv")

df["Pre-existing Conditions"] = df["Pre-existing Conditions"].fillna("None")

encoder = LabelEncoder()

for col in df.select_dtypes(include="object").columns:
    df[col] = encoder.fit_transform(df[col])

X = df.drop(["Participant_ID", "CRC_Risk"], axis=1)
y = df["CRC_Risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

baseline = LogisticRegression(max_iter=2000)
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

before_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "Precision": [
        precision_score(y_test, baseline_pred),
        precision_score(y_test, rf_pred)
    ],
    "Recall": [
        recall_score(y_test, baseline_pred),
        recall_score(y_test, rf_pred)
    ],
    "F1 Score": [
        f1_score(y_test, baseline_pred),
        f1_score(y_test, rf_pred)
    ]
})

print("BEFORE - Original Random Split")
before_results

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


BEFORE - Original Random Split


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.870,0.692308,0.290323,0.409091
1,Random Forest,0.905,0.928571,0.419355,0.577778


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In Week 5, I used an 80/20 stratified random split. Under this split, the Random Forest model achieved an observed accuracy of 90.5% and an F1-score of 57.8%.

For this audit, I checked the available grouping information before changing the validation design. The dataset contains `Participant_ID`, but it does not contain a separate client identifier or date column. Each participant is represented as an individual record.

I therefore used `Participant_ID` as the grouping key for the audit. This keeps each participant together and avoids creating a client or time grouping that does not actually exist in the dataset.

The original random split produced the following measured results for Random Forest:

* Accuracy: 90.5%
* Precision: 92.9%
* Recall: 41.9%
* F1-score: 57.8%

The grouped split is calculated below using the same Random Forest model and the same feature set.

Because each participant appears only once, the grouped split does not represent a true client-level validation. Therefore, I will not claim that this experiment proves generalisation to unseen clients. Instead, it shows how the model behaves when `Participant_ID` is respected as the grouping variable.

The purpose of this comparison is to make the validation process more transparent and identify limitations in the available data.


In [ ]:
groups = df["Participant_ID"]

group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    group_split.split(X, y, groups=groups)
)

X_group_train = X.iloc[train_idx]
X_group_test = X.iloc[test_idx]

y_group_train = y.iloc[train_idx]
y_group_test = y.iloc[test_idx]

rf_group = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_group.fit(X_group_train, y_group_train)

rf_group_pred = rf_group.predict(X_group_test)

after_accuracy = accuracy_score(
    y_group_test,
    rf_group_pred
)

after_precision = precision_score(
    y_group_test,
    rf_group_pred
)

after_recall = recall_score(
    y_group_test,
    rf_group_pred
)

after_f1 = f1_score(
    y_group_test,
    rf_group_pred
)

after_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Before Random Split": [
        accuracy_score(y_test, rf_pred),
        precision_score(y_test, rf_pred),
        recall_score(y_test, rf_pred),
        f1_score(y_test, rf_pred)
    ],
    "After Grouped Split": [
        after_accuracy,
        after_precision,
        after_recall,
        after_f1
    ]
})

after_results

,Metric,Before Random Split,After Grouped Split
0,Accuracy,0.905000,0.885000
1,Precision,0.928571,0.882353
2,Recall,0.419355,0.416667
3,F1 Score,0.577778,0.566038


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I reviewed the final feature set for possible data leakage. The main question was whether any feature contains the target directly, is derived from the target, duplicates the target information, or contains information that would not be available at the time of prediction.

I also checked the preprocessing process. The Week-5 notebook performed categorical encoding before the train-test split. Although this does not use the target values directly, it means the preprocessing step was fitted using the complete dataset. A stricter approach is to fit preprocessing only on the training data.

The final feature set does not include `CRC_Risk` or `Participant_ID`, so these two columns are not used as model predictors.

I also checked whether there are suspicious feature names that directly describe the target.


In [ ]:
target_column = "CRC_Risk"

feature_columns = X.columns.tolist()

print("Target column:", target_column)
print("Number of model features:", len(feature_columns))

print("\nModel features:")
for column in feature_columns:
    print("-", column)

direct_target_leakage = [
    column for column in feature_columns
    if column.lower() == target_column.lower()
]

print("\nDirect target leakage:", direct_target_leakage)

id_features = [
    column for column in feature_columns
    if "id" in column.lower()
]

print("\nPossible ID features:", id_features)

Target column: CRC_Risk
Number of model features: 13

Model features:
- Age
- Gender
- BMI
- Lifestyle
- Ethnicity
- Family_History_CRC
- Pre-existing Conditions
- Carbohydrates (g)
- Proteins (g)
- Fats (g)
- Vitamin A (IU)
- Vitamin C (mg)
- Iron (mg)

Direct target leakage: []

Possible ID features: []


In [ ]:
suspicious_terms = [
    "target",
    "risk",
    "label",
    "outcome",
    "diagnosis",
    "result"
]

suspicious_features = [
    column for column in feature_columns
    if any(term in column.lower() for term in suspicious_terms)
]

print("Features with potentially suspicious names:")
print(suspicious_features)

### Leakage audit result

The direct leakage check confirms that `CRC_Risk` was removed from the model features and `Participant_ID` was also excluded.

The feature names do not contain an obvious duplicate of the target variable. However, the audit identified one preprocessing improvement: categorical encoding should ideally be fitted on the training data only and then applied to the test data.

Therefore, I would not describe the current pipeline as completely leakage-free. Instead, I would describe the audit as finding no obvious direct target leakage while identifying preprocessing methodology that should be improved for a stricter validation pipeline.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### claim

In the tested dataset, the Random Forest model showed observed accuracy of 90.5% and an F1-score of 57.8% under the original random split. The model also showed higher measured performance than the Logistic Regression baseline on that split.

However, the model's recall for the high-risk class was 41.9%, meaning that some high-risk participants were not identified. The dataset also does not contain a true client identifier or time variable for stronger client-level or time-aware validation.

Therefore, the results should be treated as measured and directional evidence from this dataset. The model can be considered a decision-support prototype for this project, but the results do not provide enough evidence to claim clinical accuracy or real-world generalisation.

### What I learned from the audit

The main lesson from this audit is that a high accuracy score does not automatically mean that a model is reliable in a different setting. Validation design, feature leakage, target construction, and real error examples all affect how confidently a result can be interpreted.

For this project, I can support claims about observed performance on the tested dataset, but broader claims would require independent data and stronger validation.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.